In [4]:
import psycopg2 as ps
import pandas as pd

In [3]:
# ahora estableceremos los parámetros de conexión con la base de datos, 
# asegúrese de usar los parámetros definidos en el proceso de creación
host = "proyectobd.c10pdetlifr0.us-east-1.rds.amazonaws.com"
port = '5432'
user = 'postgres'
password = 'unisabana'
database = 'basedatosproyectobd' 

In [11]:
# Es momento de probar la conexión, 
# el resultado deberá ser Connected! si todo funcionó perfectamente

try:
    connps = ps.connect(host=host,database=database,user=user,password=password,port=port)
except ps.OperationalError as e:
    raise e
else:
    print('Connected!')

Connected!


## Prueba de carga y consulta
Creamos una tabla pequeña, insertamos 5 filas, las consultamos y al final limpiamos para no dejar nada ocupando espacio.

In [12]:
# Creamos una tabla pequeña de prueba (si ya existe, la recreamos)
with connps.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS prueba_ventas;")
    cur.execute("""
        CREATE TABLE prueba_ventas (
            id        SERIAL PRIMARY KEY,
            producto  VARCHAR(50),
            ciudad    VARCHAR(50),
            cantidad  INTEGER,
            precio    NUMERIC(10,2),
            fecha     DATE
        );
    """)
connps.commit()
print('Tabla prueba_ventas creada')

Tabla prueba_ventas creada


In [13]:
# Creamos unos pocos datos de prueba (5 filas) y los cargamos
datos = [
    ('Café',    'Bogotá',       3, 4500.0, '2026-10-01'),
    ('Arepa',   'Medellín',     5, 2500.0, '2026-10-02'),
    ('Empanada','Cali',         10, 1800.0, '2026-10-03'),
    ('Café',    'Medellín',     2, 4500.0, '2026-10-04'),
    ('Tamal',   'Bogotá',       1, 9000.0, '2026-10-05'),
]

with connps.cursor() as cur:
    cur.executemany(
        "INSERT INTO prueba_ventas (producto, ciudad, cantidad, precio, fecha) VALUES (%s, %s, %s, %s, %s);",
        datos
    )
connps.commit()
print(f'{len(datos)} filas insertadas')

5 filas insertadas


In [14]:
# Consultamos los datos cargados
df = pd.read_sql("SELECT * FROM prueba_ventas ORDER BY id;", connps)
df

C:\Users\carri\AppData\Local\Temp\ipykernel_31012\871790395.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM prueba_ventas ORDER BY id;", connps)


,id,producto,ciudad,cantidad,precio,fecha
0,1,Café,Bogotá,3,4500.0,2026-10-01
1,2,Arepa,Medellín,5,2500.0,2026-10-02
2,3,Empanada,Cali,10,1800.0,2026-10-03
3,4,Café,Medellín,2,4500.0,2026-10-04
4,5,Tamal,Bogotá,1,9000.0,2026-10-05


In [15]:
# Consulta con agregación: total vendido por ciudad
query = """
    SELECT ciudad, COUNT(*) AS num_ventas, SUM(cantidad * precio) AS total
    FROM prueba_ventas
    GROUP BY ciudad
    ORDER BY total DESC;
"""
pd.read_sql(query, connps)

C:\Users\carri\AppData\Local\Temp\ipykernel_31012\715435542.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query, connps)


,ciudad,num_ventas,total
0,Bogotá,2,22500.0
1,Medellín,2,21500.0
2,Cali,1,18000.0


In [ ]:
# Limpieza: eliminamos la tabla de prueba y cerramos la conexión
# (comente la línea del DROP si desea conservar la tabla)
#with connps.cursor() as cur:
#    cur.execute("DROP TABLE IF EXISTS prueba_ventas;")
#connps.commit()
#connps.close()
#print('Tabla eliminada y conexión cerrada')

Tabla eliminada y conexión cerrada
